In [1]:
from joint_bigru_attention_crf import (
    seed_everything,
    read_data,
    build_mappings,
    MassiveDataset,
    collate,
    JointBiGRUAttentionCRF,
    evaluate,
    decode_slots
)

print("All imports successful!")

All imports successful!


In [2]:
from pathlib import Path

DATA_PATH = Path("/Users/adityamd/Desktop/NTU/Year_4/Sem_1/NLP/1.0/data/en-US.jsonl")

splits = read_data(DATA_PATH)

train_data = splits["train"]
val_data = splits["dev"]
test_data = splits["test"]

assert len(train_data) == 11514
assert len(val_data) == 2033
assert len(test_data) == 2974

print("\nFirst training example:")
print(train_data[0])

Dataset split sizes: {'train': 11514, 'dev': 2033, 'test': 2974}

First training example:
{'id': '1', 'locale': 'en-US', 'partition': 'train', 'scenario': 'alarm', 'intent': 'alarm_set', 'utt': 'wake me up at nine am on friday', 'annot_utt': 'wake me up at [time : nine am] on [date : friday]', 'worker_id': '1'}


In [3]:
vocab, intent_to_id, tag_to_id = build_mappings(train_data)

id_to_intent = {
    idx: intent
    for intent, idx in intent_to_id.items()
}

id_to_tag = {
    idx: tag
    for tag, idx in tag_to_id.items()
}

print("\nVocabulary size:", len(vocab))
print("Number of intents:", len(intent_to_id))
print("Number of BIO tags:", len(tag_to_id))

print("\nSample intents:")
print(list(intent_to_id.items())[:10])

print("\nSample BIO tags:")
print(list(tag_to_id.items())[:10])

Vocabulary size: 5146
Number of intents: 60
Number of BIO tags: 111

Vocabulary size: 5146
Number of intents: 60
Number of BIO tags: 111

Sample intents:
[('alarm_query', 0), ('alarm_remove', 1), ('alarm_set', 2), ('audio_volume_down', 3), ('audio_volume_mute', 4), ('audio_volume_other', 5), ('audio_volume_up', 6), ('calendar_query', 7), ('calendar_remove', 8), ('calendar_set', 9)]

Sample BIO tags:
[('O', 0), ('B-alarm_type', 1), ('B-app_name', 2), ('B-artist_name', 3), ('B-audiobook_author', 4), ('B-audiobook_name', 5), ('B-business_name', 6), ('B-business_type', 7), ('B-change_amount', 8), ('B-coffee_type', 9)]


In [4]:
train_dataset = MassiveDataset(
    train_data, vocab, intent_to_id, tag_to_id
)

val_dataset = MassiveDataset(
    val_data, vocab, intent_to_id, tag_to_id
)

test_dataset = MassiveDataset(
    test_data, vocab, intent_to_id, tag_to_id
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Testing dataset:", len(test_dataset))

Training dataset: 11514
Validation dataset: 2033
Testing dataset: 2974


In [6]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate
)

batch = next(iter(train_loader))

print("Tokens:", batch["tokens"].shape)
print("Tags:", batch["tags"].shape)
print("Intent labels:", batch["intents"].shape)
print("Attention mask:", batch["mask"].shape)

Tokens: torch.Size([64, 22])
Tags: torch.Size([64, 22])
Intent labels: torch.Size([64])
Attention mask: torch.Size([64, 22])


In [7]:
import torch

seed_everything(42)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Using device:", device)

model = JointBiGRUAttentionCRF(
    vocab_size=len(vocab),
    num_intents=len(intent_to_id),
    num_tags=len(tag_to_id),
    embed_dim=100,
    hidden_size=128,
    layers=2,
    dropout=0.3
)

model = model.to(device)

total_params = sum(
    p.numel() for p in model.parameters()
)

print(f"Total parameters: {total_params:,}")

assert total_params <= 20_000_000
print("Parameter limit satisfied!")

Using device: mps
Total parameters: 1,044,435
Parameter limit satisfied!


In [8]:
batch = next(iter(train_loader))

tokens = batch["tokens"].to(device)
mask = batch["mask"].to(device)
tags = batch["tags"].to(device)
intents = batch["intents"].to(device)

model.train()

outputs = model(
    tokens=tokens,
    mask=mask,
    tags=tags,
    intents=intents
)

print("Loss:", outputs["loss"].item())
print("Intent loss:", outputs["intent_loss"])
print("Slot loss:", outputs["slot_loss"])

print(
    "Intent logits shape:",
    outputs["intent_logits"].shape
)

print(
    "Slot emissions shape:",
    outputs["emissions"].shape
)

Loss: 39.280635833740234
Intent loss: 4.1208600997924805
Slot loss: 35.15977478027344
Intent logits shape: torch.Size([64, 60])
Slot emissions shape: torch.Size([64, 17, 111])


In [9]:
import time
import json

EPOCHS = 30
LEARNING_RATE = 0.001

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

output_dir = Path("stage2_outputs")
output_dir.mkdir(exist_ok=True)

checkpoint_path = output_dir / "best_bigru_attention_crf.pt"

history = []

best_score = -1
best_epoch = -1

print("Training configuration ready!")

Training configuration ready!


In [11]:
%pip install tqdm

DEPRECATION: Loading egg at /opt/anaconda3/lib/python3.12/site-packages/yahoo_finance-1.4.0-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Note: you may need to restart the kernel to use updated packages.


In [10]:

from tqdm.auto import tqdm
import time

for epoch in range(1, EPOCHS + 1):

    start_time = time.perf_counter()
    model.train()

    total_train_loss = 0.0
    total_samples = 0

    print(f"\n{'='*55}", flush=True)
    print(f"EPOCH {epoch}/{EPOCHS}", flush=True)
    print(f"{'='*55}", flush=True)

    # Progress bar for training batches
    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{EPOCHS} Training",
        total=len(train_loader),
        unit="batch",
        leave=True
    )

    for batch in progress_bar:

        tokens = batch["tokens"].to(device)
        mask = batch["mask"].to(device)
        tags = batch["tags"].to(device)
        intents = batch["intents"].to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            tokens=tokens,
            mask=mask,
            tags=tags,
            intents=intents
        )

        loss = outputs["loss"]

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        batch_size = tokens.size(0)

        total_train_loss += loss.item() * batch_size
        total_samples += batch_size

        # Update progress bar after every batch
        progress_bar.set_postfix({
            "loss": f"{loss.item():.3f}",
            "avg_loss": f"{total_train_loss / total_samples:.3f}",
            "intent_loss": f"{outputs['intent_loss']:.3f}",
            "slot_loss": f"{outputs['slot_loss']:.3f}"
        })

    avg_train_loss = total_train_loss / total_samples

    print("\nTraining completed. Running validation...", flush=True)

    # Evaluate model on validation dataset
    val_results = evaluate(
        model,
        val_loader,
        device,
        id_to_tag
    )

    val_loss = val_results["loss"]
    intent_acc = val_results["intent_accuracy"]
    slot_f1 = val_results["slot_micro_f1"]

    selection_score = (intent_acc + slot_f1) / 2

    elapsed = time.perf_counter() - start_time

    epoch_results = {
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "val_loss": val_loss,
        "intent_accuracy": intent_acc,
        "slot_f1": slot_f1,
        "selection_score": selection_score,
        "seconds": elapsed
    }

    history.append(epoch_results)

    print(f"\n{'-'*55}", flush=True)
    print(f"EPOCH {epoch} RESULTS", flush=True)
    print(f"{'-'*55}", flush=True)
    print(f"Training Loss:       {avg_train_loss:.4f}")
    print(f"Validation Loss:     {val_loss:.4f}")
    print(f"Intent Accuracy:     {intent_acc:.2f}%")
    print(f"Slot Micro-F1:       {slot_f1:.2f}%")
    print(f"Combined Score:      {selection_score:.2f}")
    print(f"Epoch Time:          {elapsed:.2f} seconds")

    # Save best model
    if selection_score > best_score:

        best_score = selection_score
        best_epoch = epoch

        torch.save(
            {
                "model": model.state_dict(),
                "vocab": vocab,
                "intent_to_id": intent_to_id,
                "tag_to_id": tag_to_id,
                "epoch": epoch,
                "validation_score": best_score,
                "total_parameters": total_params
            },
            checkpoint_path
        )

        print("\nNEW BEST MODEL SAVED!", flush=True)

    print(
        f"Best so far: Epoch {best_epoch}, "
        f"Score {best_score:.2f}",
        flush=True
    )

print("\n" + "="*55)
print("TRAINING COMPLETE")
print("="*55)
print(f"Best epoch: {best_epoch}")
print(f"Best validation score: {best_score:.2f}")
print(f"Checkpoint: {checkpoint_path}")



EPOCH 1/30


Epoch 1/30 Training:   0%|          | 0/180 [00:00<?, ?batch/s]


Training completed. Running validation...

-------------------------------------------------------
EPOCH 1 RESULTS
-------------------------------------------------------
Training Loss:       13.3584
Validation Loss:     8.4383
Intent Accuracy:     30.99%
Slot Micro-F1:       38.01%
Combined Score:      34.50
Epoch Time:          79.17 seconds

NEW BEST MODEL SAVED!
Best so far: Epoch 1, Score 34.50

EPOCH 2/30


Epoch 2/30 Training:   0%|          | 0/180 [00:00<?, ?batch/s]


Training completed. Running validation...

-------------------------------------------------------
EPOCH 2 RESULTS
-------------------------------------------------------
Training Loss:       7.7139
Validation Loss:     5.6801
Intent Accuracy:     51.80%
Slot Micro-F1:       54.69%
Combined Score:      53.24
Epoch Time:          71.13 seconds

NEW BEST MODEL SAVED!
Best so far: Epoch 2, Score 53.24

EPOCH 3/30


Epoch 3/30 Training:   0%|          | 0/180 [00:00<?, ?batch/s]

... (51 outputs left out)

Epoch 29/30 Training:   0%|          | 0/180 [00:00<?, ?batch/s]


Training completed. Running validation...

-------------------------------------------------------
EPOCH 29 RESULTS
-------------------------------------------------------
Training Loss:       0.9648
Validation Loss:     2.5303
Intent Accuracy:     83.47%
Slot Micro-F1:       79.26%
Combined Score:      81.37
Epoch Time:          73.45 seconds

NEW BEST MODEL SAVED!
Best so far: Epoch 29, Score 81.37

EPOCH 30/30


Epoch 30/30 Training:   0%|          | 0/180 [00:00<?, ?batch/s]


Training completed. Running validation...

-------------------------------------------------------
EPOCH 30 RESULTS
-------------------------------------------------------
Training Loss:       0.9298
Validation Loss:     2.5515
Intent Accuracy:     84.01%
Slot Micro-F1:       78.98%
Combined Score:      81.50
Epoch Time:          74.38 seconds

NEW BEST MODEL SAVED!
Best so far: Epoch 30, Score 81.50

TRAINING COMPLETE
Best epoch: 30
Best validation score: 81.50
Checkpoint: stage2_outputs/best_bigru_attention_crf.pt
